# 1. Introduction

This notebook benchmarks **`ocr-resilience`**, a classical computer-vision preprocessing + multi-engine ensemble layer built *on top of* existing OCR engines (Tesseract, EasyOCR, PaddleOCR) — not a replacement for them, and not a trained model or LLM correction pass. Every technique in the package (blur/noise/skew detection, CLAHE, Sauvola binarization, ROVER-style multi-hypothesis voting) is a well-established, decades-old computer-vision method, applied only where a benchmark actually showed it helps.

This notebook is meant to run top-to-bottom and regenerate every number in it from scratch — nothing here is pasted from elsewhere without also being reproducible by re-running the cell that produced it. Numbers will differ from the project's own reference run (see the README) because Kaggle's hardware/OS/Tesseract build differ from that reference environment — that's expected, not a bug; see Section 12.

# 2. Problem Statement

Off-the-shelf OCR engines are excellent on clean, scanned documents but degrade unevenly on messy real-world input: a blurry phone photo, a skewed scan, a smudged receipt, low contrast under uneven lighting. Two open questions this notebook investigates with actual experiments rather than assertion:

1. **Does classical CV preprocessing (deskew, denoise, contrast enhancement, binarization), applied *conditionally* based on measured image quality, improve OCR accuracy over calling an engine directly — and if so, on which specific degradation types?**
2. **Does combining multiple OCR engines (via quality-aware routing + ROVER-style consensus voting) outperform any single engine, and is the extra latency worth it?**

The central philosophy (see the package README): *do not claim the pipeline is better — build the experiment that shows where it is better, where it is worse, and why.*

# 3. Dataset

No real scanned/handwritten corpus with exact ground truth was available in this environment, so the benchmark instead **renders clean text with known ground truth via PIL**, then applies controlled synthetic degradation — the same "construct exact ground truth" strategy used throughout this dataset's origin project. This gives exact per-image CER/WER (no manual transcription/labeling error), at the honest cost documented below.

- **10 sentences** covering varied character classes: digits, currency, mixed case, hyphenated codes (e.g. `"Invoice Number INV-8452"`, `"Account Balance Rs 45,231.90"`).
- **2 styles per sentence**: `printed` (a standard sans-serif font) and `handwritten_proxy` (a cursive TrueType font).
- **11 degradation presets**: clean, light/heavy Gaussian blur, motion blur, Gaussian noise, salt-and-pepper noise, rotation/skew, low contrast, smudges (inpainted-blob occlusions), JPEG compression artifacts, and a stacked "combo_hard" (rotation + blur + noise + smudge together).

**Honesty note, read before trusting the handwriting numbers below:** `handwritten_proxy` is a cursive *font*, not real handwriting. Real handwritten strokes are far more irregular than any font can produce (no consistent kerning, inconsistent pen pressure/joins) — treat every handwriting-labeled result in this notebook as a **lower bound** on real handwriting difficulty, not an equivalent test. No real handwriting corpus (e.g. IAM, which requires registration) was available in this environment.

In [ ]:
# Clone the package repo (gives us the `benchmark/` harness alongside the installable `ocr_resilience` package)
!git clone --quiet https://github.com/Abishek9342/ocr-resilience-pipeline.git
%cd ocr-resilience-pipeline
!pip install -q -e ".[tesseract,easyocr,benchmark,dev]"
# Tesseract's native binary (pytesseract is only the Python wrapper) — apt package on Kaggle's Ubuntu-based image
!apt-get -qq update && apt-get -qq install -y tesseract-ocr
!tesseract --version

In [ ]:
import sys
sys.path.insert(0, ".")

from benchmark.corpus import build_corpus, SENTENCES

manifest = build_corpus("_corpus_cache")
print(f"{len(manifest)} images ({len(SENTENCES)} sentences x 2 styles)")
manifest[:3]

# 4. Baselines

The comparison systems in this notebook: **Tesseract alone**, **EasyOCR alone** — each called directly with no preprocessing, no routing, no fusion — and **`ours`**, this package's adaptive pipeline built from whichever of those engines load successfully. PaddleOCR is attempted too; if it fails to load (a known upstream PP-OCRv6/PaddlePaddle model-loading bug affects some environments — see the package README), it is skipped and reported as skipped, not silently dropped.

In [ ]:
import cv2
import matplotlib.pyplot as plt

from ocr_resilience import OCR
from ocr_resilience.quality import assess

sample = cv2.imread(manifest[3]["path"])  # a 'printed' sentence
print("Ground truth:", manifest[3]["ground_truth"])
print(assess(sample))
plt.imshow(cv2.cvtColor(sample, cv2.COLOR_BGR2RGB)); plt.axis("off"); plt.title("Sample corpus image");

# 5. Our Methodology

The pipeline: **quality assessment** (classical CV metrics — Laplacian-variance blur, MAD-based noise estimate, pixel std-dev contrast, minAreaRect skew angle, stroke-width-variance handwriting heuristic) -> **targeted preprocessing** (deskew/denoise/deblur/CLAHE, each applied *only if the quality report says it's needed* — not a fixed always-on chain) -> **quality-aware routing** (a clean image goes to one fast engine; a hard case — multiple degradations stacked, or handwriting-like — goes to an ensemble of every available engine) -> **ROVER-style fusion** (spatial grouping via intersection-over-smaller-box-area, then confidence-weighted character-level consensus voting across engines) -> **line-aware text reconstruction**.

This is a genuinely different question from "run every engine and concatenate": the adaptive design exists specifically to avoid paying ensemble latency on easy images.

In [ ]:
ocr = OCR(engine="tesseract")  # a single fixed engine, adaptive preprocessing on
result = ocr.predict(sample)
print("Text:", result.raw_text)
print("Preprocessing applied:", result.preprocessing_pipeline)
print("Routing:", result.routing.reason)
print("Confidence:", round(result.confidence, 3))

# 6. Preprocessing Experiments

Before trusting the adaptive pipeline's design, does each individual preprocessing operator actually help — or does *some* of them hurt? The cell below runs the same forced-single-technique comparison the project's own ablation study uses (`OCRPipeline.run(force_step=...)`), on a heavily-blurred sample, so you can see the effect directly rather than take the aggregate ablation table's word for it.

In [ ]:
from benchmark.degrade import apply_degradation
from ocr_resilience.pipeline import OCRPipeline
from ocr_resilience.metrics import cer

truth = manifest[3]["ground_truth"]
degraded = apply_degradation(sample, "heavy_blur", seed=0)
pipeline = OCRPipeline.with_engines(["tesseract"])

for label, kwargs in [
    ("no preprocessing", dict(skip_preprocessing=True)),
    ("+ deskew", dict(force_step="deskew")),
    ("+ denoise", dict(force_step="denoise")),
    ("+ deblur (unsharp)", dict(force_step="deblur_unsharp")),
    ("+ CLAHE contrast", dict(force_step="enhance_contrast")),
    ("+ Sauvola binarization", dict(force_step="binarize_sauvola")),
    ("adaptive (quality-gated chain)", dict(skip_preprocessing=False)),
]:
    r = pipeline.run(degraded, force_ensemble=False, **kwargs)
    print(f"{label:32s} CER={cer(r.raw_text, truth):.4f}  text={r.raw_text!r}")

# 7. OCR Experiments

Now the multi-engine question: does running Tesseract + EasyOCR together and fusing beat either alone, and by how much does latency grow?

In [ ]:
pipeline2 = OCRPipeline.with_engines(["tesseract", "easyocr"])
for label, kwargs in [("single engine (tesseract only)", dict(force_ensemble=False)), ("forced ensemble (both engines + fusion)", dict(force_ensemble=True))]:
    import time
    t0 = time.perf_counter()
    r = pipeline2.run(degraded, skip_preprocessing=True, **kwargs)
    dt = time.perf_counter() - t0
    print(f"{label:42s} CER={cer(r.raw_text, truth):.4f}  latency={dt:.3f}s  text={r.raw_text!r}")

# 8. Benchmark Results

Full run across all 20 corpus images x all 11 degradation presets x {tesseract, easyocr, ours} (paddleocr included in `--engines` but skipped automatically if it fails to load in this environment — see the printed `[skipped]` line if so). This is the same `benchmark/run_benchmark.py` CLI documented in the package README's Benchmark section; it writes `benchmark/results/{raw_results.csv,summary.csv,latency.csv,benchmark.json}`.

In [ ]:
!python -m benchmark.run_benchmark --engines tesseract,easyocr,paddleocr,ours --presets all

In [ ]:
import pandas as pd

summary = pd.read_csv("benchmark/results/summary.csv", index_col=0)
summary

In [ ]:
raw = pd.read_csv("benchmark/results/raw_results.csv")
pivot = raw.pivot_table(index="preset", columns="system", values="cer", aggfunc="mean").round(4)
pivot.plot(kind="bar", figsize=(11, 5), title="Mean CER by degradation preset and system")
plt.ylabel("Character Error Rate (lower is better)"); plt.tight_layout();

**Reference numbers** (from the package's own reference run — Windows, Python 3.13, local Tesseract 5.4.0 binary + EasyOCR 1.7.2; see the README/`docs/benchmark_report.md` for exact versions and re-run instructions): the pipeline had the best *overall* mean CER (0.054 vs. 0.118 for EasyOCR alone and 0.166 for Tesseract alone), but it did **not** win every category — it won clearly on heavy blur/skew/salt-and-pepper noise, tied the better baseline on clean/light-blur/smudged/JPEG-compressed, but lost narrowly to plain Tesseract on general noise/low-contrast and clearly on motion blur, and lost clearly to plain EasyOCR on the stacked `combo_hard` case. Its actual edge is avoiding catastrophic single-engine failures (Tesseract returned completely empty output on salt-and-pepper noise; EasyOCR nearly failed on motion blur) rather than being the single best performer everywhere. **Your numbers above will differ** — different Tesseract build/version, different CPU, different EasyOCR model cache warm state — and that's the point of re-running rather than trusting a pasted table; what should NOT differ qualitatively is this pattern of wins/ties/losses.

# 9. Error Analysis

Which specific cases does each system get most wrong? Rather than only look at averages, inspect the worst individual rows.

In [ ]:
worst = raw.sort_values("cer", ascending=False).head(15)
worst[["style", "preset", "system", "cer", "wer", "latency_sec"]]

In [ ]:
# Which preset x style combination is hardest, per system?
raw.groupby(["style", "preset", "system"])["cer"].mean().unstack("system").sort_values("ours" if "ours" in raw["system"].unique() else raw["system"].unique()[0], ascending=False).head(10)

**Expected pattern** (from the reference run): `handwritten_proxy` combined with any degradation is the hardest category for every system — consistent with the corpus's own honesty note that the cursive-font proxy is already harder than print, before any degradation is even applied. If your run instead shows a *degraded printed* case as hardest, that's worth a closer look (open the specific image via its path in `raw_results.csv`'s corresponding manifest entry) rather than assumed to match this note.

# 10. Ablation Study

Which pipeline component is actually responsible for any improvement over the no-preprocessing baseline? `benchmark/run_ablation.py` measures each cell below as an independent, real run (via `OCRPipeline.run()`'s `skip_preprocessing`/`force_step`/`force_ensemble` hooks — see Section 6/7 above for the same hooks used interactively).

In [ ]:
!python -m benchmark.run_ablation

In [ ]:
ablation_summary = pd.read_csv("benchmark/results/ablation_summary.csv", index_col=0)
ablation_summary

**Reference result** (package's own run, 20 images x {clean, heavy_blur, skewed, noisy, combo_hard} x 8 variants = 800 measured cells):

| Variant | Mean CER | Mean WER | Mean latency (s) |
|---|---:|---:|---:|
| baseline (no preprocessing, single engine) | 0.0898 | 0.2475 | 0.134 |
| + deskew | 0.0653 | 0.2007 | 0.124 |
| + denoise | 0.1111 | 0.2755 | 0.222 |
| + CLAHE contrast | 0.1602 | 0.3278 | 0.121 |
| + Sauvola binarization | 0.0889 | 0.2223 | 0.117 |
| + adaptive preprocessing (quality-gated chain) | 0.0503 | 0.1925 | 0.153 |
| + multi-engine selection (forced ensemble) | 0.0320 | 0.1837 | 0.399 |
| full pipeline (adaptive + multi-engine) | 0.0350 | 0.1813 | 0.309 |

**Honest reading of this table, not a marketing one:**
- Deskew and Sauvola binarization *individually* help versus doing nothing.
- **Denoise and CLAHE, forced unconditionally on every image regardless of whether it's actually noisy/low-contrast, each made results *worse* on average** (0.111 and 0.160 CER, both above the 0.090 baseline) — this is exactly why the adaptive pipeline gates each operator on the quality report instead of always applying it, and this ablation is the evidence for that design choice, not an assumption.
- The gated **adaptive combination (0.050) beats every individual forced operator alone**, including the ones that helped in isolation — the components interact, and gating matters as much as the operators themselves.
- **Multi-engine selection is the single largest lever** (0.032, ~3x lower than baseline) but also by far the most expensive (~3x baseline latency) — the full pipeline's adaptive routing exists to only pay that cost on images that need it, which is why its latency (0.31s) sits below forced-ensemble-always (0.40s) at nearly the same accuracy.
- **Post-processing (whitespace/unicode normalization) showed no measurable CER difference** on this rendered, already-clean-whitespace synthetic corpus — an honest null result, not evidence it never helps (a real scanned document with genuine OCR whitespace artifacts is a more realistic test of that specific claim, and isn't what this corpus models).

# 11. Conclusion

On this synthetic, controlled-degradation benchmark: the adaptive pipeline had the best *overall* mean CER, and won clearly on heavy blur, skew, and salt-and-pepper noise — exactly the failure modes it was designed to target — but it did **not** win every category: plain Tesseract alone was competitive-to-better on general noise/low-contrast and clearly better on motion blur, and plain EasyOCR alone was clearly better on the stacked `combo_hard` case. Its real advantage, per the numbers above, is avoiding the catastrophic single-engine failures each baseline has somewhere (Tesseract's complete blank output on salt-and-pepper noise; EasyOCR's near-failure on motion blur) — consistency, not uniform superiority. The ablation study shows *why* the wins happen where they happen: quality-gated preprocessing selection and multi-engine consensus each contribute independently, and naively forcing every preprocessing operator on unconditionally (rather than gating on measured quality) made results *worse*, not better, for two of the five operators tested.

**What this notebook does NOT show:** performance on real scanned documents, real photographs of receipts/forms, real handwriting (vs. the cursive-font proxy), or non-Latin scripts — none of these were available in this environment. It also doesn't explain *why* fusion loses on `combo_hard`/`motion_blur` specifically (a confidence-calibration mismatch between engines is the leading hypothesis, not yet tested — see Roadmap). Treat every number here as evidence about *this specific synthetic benchmark*, not a universal claim; the package README's Limitations section states this explicitly, and extending this benchmark to a real-document dataset is the most valuable next contribution someone could make (see Roadmap).

# 12. Reproducibility Instructions

This entire notebook is designed to be re-run top to bottom with no manual steps — Section 3's clone-and-install cell is the only setup required. To reproduce outside Kaggle:

```bash
git clone https://github.com/Abishek9342/ocr-resilience-pipeline.git
cd ocr-resilience-pipeline
pip install -e ".[all,benchmark,dev]"
# Tesseract's native binary is a separate install — see README Installation section
pytest tests/                                    # 55 unit/regression tests, no OCR binary required (mocked adapters)
python -m benchmark.run_benchmark --presets all   # full benchmark -> benchmark/results/
python -m benchmark.run_ablation                  # ablation study -> benchmark/results/ablation_*.csv
```

Numbers will differ from both this notebook's own output and the README's reference table whenever the environment differs (OS, Tesseract build/version, EasyOCR/PaddleOCR model versions, CPU) — this is expected. What should NOT differ qualitatively: deskew/Sauvola helping, denoise/CLAHE forced-unconditionally hurting, and multi-engine selection being the largest single lever. If your re-run shows a qualitatively different pattern, that itself is a useful finding worth opening an issue about (see CONTRIBUTING.md in the repo).